In [ ]:
import sys, os, json, torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
warnings = __import__('warnings')
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('../../..'))

In [ ]:
from DL.models.custom_classifier_from_config import CustomClassifierFromConfig
from DL.trainers.model_trainer import ModelTrainer
from DL.visualization.plot_utils import plot_confusion_matrices
from DL.visualization.classic_gradcam import visualize_classic_gradcam
from DL.data.combined_dataset import CombinedDataset

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device: ', device)

In [ ]:
perspective_transform = T.RandomPerspective(distortion_scale=0.5, p=0.8)

train_dataset = CombinedDataset(75000, 42, transform=perspective_transform)
test_dataset = CombinedDataset(25000, 123, transform=perspective_transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
def run_test(config_name, alpha, epochs=15):
    with open(f'../../../DL/configs/lenet/{config_name}', 'r') as f:
        config = json.load(f)
    config['num_classes'] = len(train_dataset.char_to_idx)

    model = CustomClassifierFromConfig(config).to(device)
    trainer = ModelTrainer(model, torch.optim.Adam(model.parameters(), lr=1e-3), nn.CrossEntropyLoss(), device, use_background_loss=True, bg_loss_alpha=alpha, bg_loss_n=2)
    trainer.fit(train_loader, test_loader, epochs=epochs)

    plot_confusion_matrices(model, model, test_loader, test_dataset, device)
    visualize_classic_gradcam(model, test_dataset, device, f"Model: {config_name}")

In [ ]:
print("=== ЭКСПЕРИМЕНТ 1: 4KB Combined ===")
run_test('lenet_4kb.json', alpha=15.0)

In [ ]:
print("\n=== ЭКСПЕРИМЕНТ 2: 2KB Combined ===")
run_test('lenet_2kb.json', alpha=5.0) # Сниженная альфа для 2КБ